# DART 재무데이터 분석 v4 (korea_fs_data_from_DART_V3 + korea_dart_corp_master)

`dart_fs_analyzer_v4.py` 사용. 로더 v4 의 Cell 2-1 로 `korea_dart_corp_master` 를 먼저 만들어 두면
- 기업명이 DART 전 상장사 기준으로 채워지고
- 비12월 결산 법인(3·6·9월)의 분기가 달력 분기로 정렬되며
- 진단에서 DataGuide 전용 코드(우선주 등)가 `not_in_dart` 로 분리됩니다.

금액 단위 원 (`unit=1e8` → 억원). 스크리너 입력은 `panel`.

In [61]:
# ==========================================================
# PART 1: 환경 + Control Panel + DB
# ==========================================================
import sys
from pathlib import Path
import pandas as pd

def add_repo_path():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'DATA').exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            return parent
    raise FileNotFoundError('DATA 폴더를 찾을 수 없습니다')

PROJECT_ROOT = add_repo_path()
sys.path.insert(0, str(Path.cwd()))

from DATA import config
import dart_fs_analyzer_v4 as D

engine = config.get_engine(config.get_db_info())

# ---- Control Panel ----
ASOF        = None          # 기준 분기. 예: '2026Q2'. None → 최근 완전 적재 분기 자동 선택
START_YEAR  = 2019          # 패널 구축 시작 연도 (TTM/평균 계산용 여유 포함)
TICKERS     = None          # None=전체, 또는 ['A278470','A000660'] 로 제한
MISSING_ADD = 'cumulative'  # H1/Q3 IS 에 thstrm_add_amount 없을 때 thstrm_amount 해석: 'cumulative' / 'quarter'
TOP_N       = 30
UNIT        = 1e8           # 원 → 억원
PANEL_CACHE = Path('dart_panel_cache.parquet')   # None 이면 캐시 안 씀

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. 적재된 계정 목록 (키워드 검색)
`mapped` 컬럼 = 어떤 concept 으로 매핑되는지. 빈칸이면 미매핑 → 필요 시 `D.CONCEPTS` 에 id/이름 추가.

In [62]:
D.search_accounts(engine, '영업이익')
# D.search_accounts(engine, 'Revenue')
# D.search_accounts(engine, sj_div='BS', min_tickers=100)   # BS 에서 100개 종목 이상 쓰는 계정

,mapped,sj_div,account_id,account_nm,n_ticker,n_rows,min_year,max_year
0,,CF,ifrs-full_ProfitLossFromContinuingOperations,계속영업이익(손실),11,37,2025,2026
1,,CF,dart_AdjustmentsForProfitLossFromDiscontinuedO...,중단영업이익(손실) 조정,4,12,2025,2026
2,,CF,ifrs-full_ProfitLossFromContinuingOperations,계속영업이익,3,6,2025,2026
3,,CF,dart_OperatingIncomeLoss,영업이익(손실),2,3,2026,2026
4,,CF,-표준계정코드 미사용-,영업이익(손실),1,1,2026,2026
...,...,...,...,...,...,...,...,...
246,,IS,dart_DilutedEarningsLossPerSharePreferredStock...,구형우선주희석주당계속영업이익,1,1,2026,2026
247,,IS,ifrs-full_BasicEarningsLossPerShareFromContinu...,보통주 기본 및 희석주당계속영업이익(손실),1,1,2026,2026
248,,IS,ifrs-full_OtherOperatingIncomeExpense,기타 영업이익(비용),1,1,2026,2026
249,,SCE,ifrs-full_ProfitLoss,계속영업이익(손실),1,2,2026,2026


In [63]:
# 미매핑 계정 중 많이 쓰이는 것 확인 (매핑 보강 대상)
acc = D.search_accounts(engine, min_tickers=30)
acc[acc['mapped'] == ''].head(40)

,mapped,sj_div,account_id,account_nm,n_ticker,n_rows,min_year,max_year
4,,BS,ifrs-full_IssuedCapital,자본금,2788,15837,2025,2026
8,,BS,ifrs-full_NoncurrentAssets,비유동자산,2717,13296,2025,2026
9,,BS,ifrs-full_PropertyPlantAndEquipment,유형자산,2700,15501,2025,2026
10,,BS,ifrs-full_EquityAndLiabilities,자본과부채총계,2422,12918,2025,2026
11,,BS,ifrs-full_OtherCurrentAssets,기타유동자산,2335,11448,2025,2026
15,,BS,ifrs-full_NoncurrentAssets,비유동 자산,2294,2299,2025,2026
16,,BS,ifrs-full_CurrentTaxAssets,당기법인세자산,2254,10204,2025,2026
17,,BS,dart_CapitalSurplus,자본잉여금,2055,11239,2025,2026
18,,BS,ifrs-full_CurrentTaxLiabilities,당기법인세부채,2016,9209,2025,2026
19,,BS,ifrs-full_RetainedEarnings,이익잉여금(결손금),1998,9914,2025,2026


## 2. 분기 패널 구축 (1회)
원본 → concept 매핑 → 분기화(Q4 = FY − Q3누적). 전 종목이면 수 분 소요 → parquet 캐시.

In [64]:
if PANEL_CACHE and PANEL_CACHE.exists() and TICKERS is None:
    panel = pd.read_parquet(PANEL_CACHE)
    panel['q'] = pd.PeriodIndex(panel['q'], freq='Q')
    print(f'cache load: {PANEL_CACHE} ({len(panel):,} rows)')
else:
    panel = D.build_quarterly_panel(engine, tickers=TICKERS, start_year=START_YEAR, missing_add=MISSING_ADD)
    if PANEL_CACHE and TICKERS is None:
        panel.assign(q=panel['q'].astype(str)).to_parquet(PANEL_CACHE, index=False)
        print(f'cache save: {PANEL_CACHE}')
print(panel.shape, panel['ticker'].nunique(), '종목')
D.coverage_report(panel)

cache load: dart_panel_cache.parquet (424,634 rows)
(424634, 9) 2858 종목


,n_ticker,n_rows,name_match_%,add_missing_%,cis_%,max_q
concept,,,,,,
자본,2858,16383,0.05,0.00,0.00,2026Q2
부채,2858,16384,0.00,0.00,0.00,2026Q2
자산,2857,16382,0.00,0.00,0.00,2026Q2
세전이익,2820,16254,0.12,0.00,92.96,2026Q2
현금,2815,16123,0.17,0.00,0.00,2026Q2
당기순이익,2811,16166,0.22,0.00,92.93,2026Q2
영업현금흐름,2805,16161,0.14,49.77,0.00,2026Q2
투자현금흐름,2795,16076,0.08,49.74,0.00,2026Q2
재무현금흐름,2785,15985,0.08,49.75,0.00,2026Q2


`add_missing_%` 가 IS 계정에서 높으면 `MISSING_ADD` 해석이 결과를 좌우 → 아래 4번 DataGuide 대조로 확인.

## 3. 종목별 시계열

In [ ]:
# 기업 마스터 확인 (로더 v4 Cell 2-1 실행 후)
cm = D.load_corp_master(engine)
print(len(cm), '개 상장사 /  결산월 분포:')
print(cm['acc_mt'].value_counts().to_string())
# 패널을 이미 만들어 둔 경우 이름만 보충
# panel = D.fill_names(panel, engine)

In [65]:
ts = D.get_ts(panel, 'A278470', ['매출액', '매출총이익', '영업이익', '당기순이익', '자산', '자본', '영업현금흐름'], start='2023Q1', unit=UNIT)
print(ts.attrs)
ts

{'ticker': 'A278470', 'company_name': '에이피알'}


concept,매출액,매출총이익,영업이익,당기순이익,자산,자본,영업현금흐름
q,,,,,,,
2025Q1,"2,660.33","2,008.65",545.68,499.41,"5,811.91","3,455.23",535.10
2025Q2,"3,277.35","2,497.84",845.52,663.09,"6,513.37","4,081.32",656.03
2025Q3,"3,859.43","2,960.47",961.28,746.34,"6,171.30","3,487.10",658.64
2025Q4,"5,476.35","4,239.21","1,302.72",987.70,"7,717.38","4,458.00","1,560.69"
2026Q1,"5,933.56","4,569.13","1,522.72","1,172.64","9,529.96","5,065.80",42.71
2026Q2,"7,675.33","6,077.86","1,905.53","1,415.30","11,020.86","6,480.34",821.33


In [66]:
# 분기화 근거 확인 (src: q1 / add / add-missing / fy / bal,  map_src: id0.. / name / derived)
D.get_ts_with_src(panel, 'A278470', '영업이익').tail(8)

,value,src,map_src,fs_div
q,,,,
2025Q1,"54,568,141,909.00",q1,id0,CFS
2025Q2,"84,552,442,450.00",3m,id0,CFS
2025Q3,"96,127,667,281.00",3m,id0,CFS
2025Q4,"130,272,377,294.00",fy,id0,CFS
2026Q1,"152,271,694,994.00",q1,id0,CFS
2026Q2,"190,553,013,038.00",3m,id0,CFS


## 4. DataGuide 대조 (검증)
동일 종목·분기에서 DART 분기값 vs DataGuide 값. `diff%` 가 0 근처면 분기화·매핑 정상.

In [67]:
cmp = D.compare_with_dataguide(panel, engine, 'A278470', unit=UNIT)
cmp[[c for c in cmp.columns if c.endswith('diff%')]].tail(8)

,매출액_diff%,영업이익_diff%,당기순이익_diff%,자산_diff%,자본_diff%,영업현금흐름_diff%
q,,,,,,
2024Q3,NaN,NaN,NaN,NaN,NaN,NaN
2024Q4,NaN,NaN,NaN,NaN,NaN,NaN
2025Q1,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
2025Q2,-0.00,0.00,0.00,0.00,-0.00,0.00
2025Q3,0.00,0.00,0.00,-0.00,0.00,0.00
2025Q4,-0.00,-0.00,0.00,0.00,0.00,0.00
2026Q1,-0.00,4.61,0.00,-0.00,-0.00,0.00
2026Q2,-0.00,2.75,-0.00,-0.00,0.00,-0.00


## 4-1. DataGuide 대비 누락 종목 진단
DataGuide 에 t·t-4 값이 있는데 DART 패널에 없는 종목을 원인별로 분류.
- `not_in_dart` → DART 상장사 마스터에 없는 티커(DataGuide 전용 우선주·구주 코드) → 무시
- `no_report` → `dart_report` 열의 (사업연도, 보고서) 가 DART 에 없음 → 미제출이거나 수집 누락. `acc_mt`≠12 면 비12월 결산 법인이라 제출 시기가 다름
- `unmapped` → 계정명 매핑 실패 → `candidate_accounts` 보고 `D.CONCEPTS[...]['names']` 보강
- `quarterize_nan` → 매핑됐으나 분기값 산출 실패 → `D.trace()`

In [68]:
diag = D.diagnose_missing(engine, panel, item='매출액', dg_item_code='M000904001', asof=ASOF)
diag

reason
no_report         23
quarterize_nan    11


,ticker,company_name,missing_q,reason,n_rows_report,candidate_accounts
0,A0008Z0,,2026Q2,no_report,0,
1,A0008Z0,,2025Q2,no_report,0,
2,A0015G0,,2026Q2,no_report,0,
3,A0015G0,,2025Q2,no_report,0,
4,A0015S0,,2026Q2,no_report,0,
5,A0015S0,,2025Q2,no_report,0,
6,A002630,오리엔트바이오,2026Q2,no_report,0,
7,A018500,동원금속,2026Q2,no_report,0,
8,A061090,세나테크놀로지,2025Q2,no_report,0,
9,A092440,기신정기,2026Q2,no_report,0,


In [69]:
# unmapped 종목의 실제 계정 확인 → 매핑 보강
for tk in diag.loc[diag['reason'] == 'unmapped', 'ticker'].unique()[:5]:
    print('=' * 30, tk)
    print(D.raw_rows(engine, tk, 2026, sj_div=['IS', 'CIS'], keyword='수익').head(10).to_string(index=False))

In [70]:
# 영업이익도 동일 진단
D.diagnose_missing(engine, panel, item='영업이익', dg_item_code='M000906001', asof=ASOF)

reason
no_report         22
quarterize_nan    12


,ticker,company_name,missing_q,reason,n_rows_report,candidate_accounts
0,A0008Z0,,2026Q2,no_report,0,
1,A0008Z0,,2025Q2,no_report,0,
2,A0015G0,,2026Q2,no_report,0,
3,A0015G0,,2025Q2,no_report,0,
4,A0015S0,,2026Q2,no_report,0,
5,A0015S0,,2025Q2,no_report,0,
6,A002630,오리엔트바이오,2026Q2,no_report,0,
7,A018500,동원금속,2026Q2,no_report,0,
8,A061090,세나테크놀로지,2025Q2,no_report,0,
9,A092440,기신정기,2026Q2,no_report,0,


## 5. YoY / QoQ 상위 N

In [71]:
D.yoy_screen(panel, '매출액', n=TOP_N, asof=ASOF, unit=UNIT)

,ticker,company_name,base_q,t_q,매출액(t-4),매출액(t),growth_%
0,A250030,,2025Q2,2026Q2,123.14,"174,760.99","141,825.82"
1,A255440,야스,2025Q2,2026Q2,61.48,872.20,"1,318.79"
2,A320000,,2025Q2,2026Q2,13.91,141.08,914.08
3,A317530,,2025Q2,2026Q2,23.62,194.29,722.46
4,A052300,,2025Q2,2026Q2,21.98,150.08,582.77
5,A354200,,2025Q2,2026Q2,18.48,124.52,573.76
6,A000040,KR모터스,2025Q2,2026Q2,45.36,263.48,480.81
7,A080220,제주반도체,2025Q2,2026Q2,510.67,"2,899.38",467.76
8,A039200,오스코텍,2025Q2,2026Q2,100.17,526.49,425.61
9,A212710,,2025Q2,2026Q2,33.61,168.50,401.39


In [55]:
D.yoy_screen(panel, '영업이익', n=TOP_N, asof=ASOF, unit=UNIT, min_base=5e8)   # 기준 영업이익 ≥ 5억원

,ticker,company_name,base_q,t_q,영업이익(t-4),영업이익(t),growth_%
0,A250030,,2025Q2,2026Q2,12.45,"22,682.42","182,122.26"
1,A016450,한세예스24홀딩스,2025Q2,2026Q2,5.40,403.93,"7,376.37"
2,A353200,대덕전자,2025Q2,2026Q2,18.66,702.76,"3,665.96"
3,A003670,포스코퓨처엠,2025Q2,2026Q2,7.73,266.89,"3,351.17"
4,A080220,제주반도체,2025Q2,2026Q2,43.38,"1,218.71","2,709.48"
5,A005950,이수화학,2025Q2,2026Q2,39.09,965.59,"2,370.21"
6,A034730,SK,2025Q2,2026Q2,"1,995.54","48,412.47","2,326.03"
7,A011070,LG이노텍,2025Q2,2026Q2,113.92,"2,457.54","2,057.25"
8,A005930,삼성전자,2025Q2,2026Q2,"46,760.57","894,924.12","1,813.84"
9,A000020,동화약품,2025Q2,2026Q2,6.14,102.90,"1,575.54"


In [56]:
D.qoq_screen(panel, '매출액', n=TOP_N, asof=ASOF, unit=UNIT)

,ticker,company_name,base_q,t_q,매출액(t-1),매출액(t),growth_%
0,A250030,,2026Q1,2026Q2,134.84,"174,760.99","129,508.41"
1,A039200,오스코텍,2026Q1,2026Q2,36.48,526.49,"1,343.13"
2,A000040,KR모터스,2026Q1,2026Q2,25.55,263.48,931.15
3,A372910,한컴라이프케어,2026Q1,2026Q2,59.89,550.74,819.62
4,A475150,SK이터닉스,2026Q1,2026Q2,275.23,"2,408.92",775.23
5,A200350,,2026Q1,2026Q2,12.55,100.40,699.91
6,A412540,,2026Q1,2026Q2,154.59,"1,116.17",622.04
7,A365270,큐라클,2026Q1,2026Q2,10.49,71.85,584.91
8,A290520,,2026Q1,2026Q2,13.40,76.19,468.41
9,A461300,아이스크림미디어,2026Q1,2026Q2,152.97,771.84,404.59


In [57]:
D.qoq_screen(panel, '영업이익', n=TOP_N, asof=ASOF, unit=UNIT)

,ticker,company_name,base_q,t_q,영업이익(t-1),영업이익(t),growth_%
0,A250030,,2026Q1,2026Q2,11.92,"22,682.42","190,164.50"
1,A035760,CJ ENM,2026Q1,2026Q2,14.60,334.47,"2,191.43"
2,A036830,솔브레인홀딩스,2026Q1,2026Q2,14.92,318.92,"2,037.70"
3,A042700,한미반도체,2026Q1,2026Q2,84.56,"1,303.47","1,441.42"
4,A036190,금화피에스시,2026Q1,2026Q2,12.84,168.24,"1,209.77"
5,A010060,OCI홀딩스,2026Q1,2026Q2,108.62,"1,080.86",895.05
6,A130580,,2026Q1,2026Q2,12.59,110.78,779.70
7,A000100,유한양행,2026Q1,2026Q2,88.15,668.69,658.57
8,A000390,SP삼화,2026Q1,2026Q2,18.21,137.51,655.23
9,A161000,애경케미칼,2026Q1,2026Q2,58.16,428.45,636.65


## 6. 영업이익 흑자전환

In [58]:
D.turnaround_screen(panel, basis='yoy', asof=ASOF, unit=UNIT)

,ticker,company_name,base_q,t_q,영업이익(t-4),영업이익(t),swing,매출액(t),swing_%rev
0,A096770,SK이노베이션,2025Q2,2026Q2,"-4,175.69","34,872.96","39,048.65","291,572.05",13.39
1,A010950,S-Oil,2025Q2,2026Q2,"-3,439.71","9,650.15","13,089.86","113,434.88",11.54
2,A006400,삼성SDI,2025Q2,2026Q2,"-3,978.37","2,037.75","6,016.11","37,688.08",15.96
3,A011170,롯데케미칼,2025Q2,2026Q2,"-2,448.80","1,101.39","3,550.19","56,863.70",6.24
4,A010060,OCI홀딩스,2025Q2,2026Q2,-803.42,"1,080.86","1,884.28","10,231.83",18.42
...,...,...,...,...,...,...,...,...,...
274,A222980,,2025Q2,2026Q2,-0.03,2.76,2.79,269.44,1.03
275,A143540,,2025Q2,2026Q2,-1.36,1.30,2.66,165.58,1.61
276,A004410,서울식품,2025Q2,2026Q2,-0.05,2.24,2.30,168.43,1.36
277,A057540,,2025Q2,2026Q2,-0.65,0.77,1.42,242.21,0.58


In [59]:
D.turnaround_screen(panel, basis='qoq', asof=ASOF, unit=UNIT)

,ticker,company_name,base_q,t_q,영업이익(t-1),영업이익(t),swing,매출액(t),swing_%rev
0,A051910,LG화학,2026Q1,2026Q2,-496.91,"5,995.88","6,492.79","141,759.01",4.58
1,A352820,하이브,2026Q1,2026Q2,"-1,965.75","1,709.25","3,674.99","14,499.98",25.34
2,A006400,삼성SDI,2026Q1,2026Q2,"-1,555.81","2,037.75","3,593.55","37,688.08",9.53
3,A373220,LG에너지솔루션,2026Q1,2026Q2,"-2,077.55","1,133.02","3,210.57","75,602.49",4.25
4,A017940,E1,2026Q1,2026Q2,"-1,561.89","1,178.99","2,740.87","44,028.08",6.23
...,...,...,...,...,...,...,...,...,...
256,A237820,,2026Q1,2026Q2,-0.98,0.89,1.87,111.20,1.68
257,A285800,,2026Q1,2026Q2,-0.90,0.79,1.69,85.93,1.96
258,A270870,,2026Q1,2026Q2,-0.26,0.07,0.33,283.62,0.12
259,A008700,아남전자,2026Q1,2026Q2,-0.03,0.01,0.04,0.45,9.03


## 7. 재무비율 스크리너
`OPM, NPM, GPM, ROE, ROE_지배, ROA, ROIC, 부채비율, 순차입금비율, OCF_margin, FCF_margin`

- 손익 TTM 합, BS 기초/기말 평균, ROIC = OP×(1−유효세율)/(자본+차입금−현금성)
- DART 는 지배주주순이익·지배주주지분이 있어 `ROE_지배` 추가

In [60]:
ratios = D.compute_ratios(panel, asof=ASOF, ttm=True, unit=UNIT)
ratios.shape

(2858, 30)

In [ ]:
D.ratio_screen(None, 'ROE', n=TOP_N, ratios_df=ratios, unit=UNIT)

In [ ]:
D.ratio_screen(None, 'ROE_지배', n=TOP_N, ratios_df=ratios, unit=UNIT)

In [ ]:
D.ratio_screen(None, 'ROIC', n=TOP_N, ratios_df=ratios, unit=UNIT)

In [ ]:
D.ratio_screen(None, 'OPM', n=TOP_N, ratios_df=ratios, unit=UNIT)

In [ ]:
D.ratio_screen(None, '부채비율', n=TOP_N, ratios_df=ratios, unit=UNIT, ascending=True)

In [ ]:
ratios[ratios['ticker'].isin(['A278470', 'A000660', 'A004000'])].T

## 8. DataGuide 미반영 최신 분기만 보기
DART 최신 분기가 DataGuide 최신 분기보다 앞서 있는 종목 = 시차 극복 대상.

In [ ]:
from sqlalchemy import text
with engine.connect() as con:
    dg_last = pd.read_sql(text("SELECT ticker, MAX(date) AS dg_max FROM korea_fs_data_from_DG WHERE item_code='M000904001' GROUP BY ticker"), con)
dg_last['dg_q'] = pd.to_datetime(dg_last['dg_max']).dt.to_period('Q')
dart_last = panel[panel['concept'] == '매출액'].groupby('ticker')['q'].max().rename('dart_q').reset_index()
gap = dart_last.merge(dg_last[['ticker', 'dg_q']], on='ticker', how='left')
gap = gap[gap['dart_q'] > gap['dg_q'].fillna(pd.Period('1900Q1', freq='Q'))]
print(f'DART 가 DataGuide 보다 앞선 종목: {len(gap)}개')
gap.head(30)